In [ ]:
!gsutil -m cp   "gs://igvf-pertub-seq-pipeline-data/scratch/bioinfolucas/Benchmark_cleanser_unified/Engreitz_ccPerturb/pipeline_dashboard" .

InvalidUrlError: Unrecognized scheme " gs".


In [ ]:
!gsutil -m cp -r "gs://igvf-pertub-seq-pipeline-data/scratch/bioinfolucas/FirstBenchmarkRun_sceptre_v11/Engreitz_ccPerturb/additional/additional_qc/gene"  .

If you experience problems with multiprocessing on MacOS, they might be related to https://bugs.python.org/issue33725. You can disable multiprocessing by editing your .boto config or by adding the following flag to your command: `-o "GSUtil:parallel_process_count=1"`. Note that multithreading is still available even if you disable multiprocessing.

ServiceException: 401 Anonymous caller does not have storage.objects.get access to the Google Cloud Storage object. Permission 'storage.objects.get' denied on resource (or it may not exist).
CommandException: 1 file/object could not be transferred.


In [ ]:


# Or, if you are in a standard Jupyter environment, run this in your terminal/cell:
!gcloud auth login

Index(['R1_path', 'R1_md5sum', 'R2_path', 'R2_md5sum', 'file_modality',
       'file_set', 'measurement_sets', 'sequencing_run', 'lane', 'flowcell_id',
       'index', 'seqspec', 'barcode_onlist', 'onlist_method',
       'strand_specificity', 'guide_design', 'barcode_hashtag_map'],
      dtype='object')

In [58]:
cd /Users/lf588/Downloads/crispr_validator/charles_gemx_v3

/Users/lf588/Downloads/crispr_validator/charles_gemx_v3


In [59]:
import os
import pandas as pd
import pandas as pd
SAMPLESHEET = '/Users/lf588/Downloads/crispr_validator/charles_gemx_v3/sample_metadata_gcp_2026_02_15.csv'
df = pd.read_csv(SAMPLESHEET)
df.columns
# --- 1. Helper Function ---
def sanitizing(x):
    """Checks if a file path contains common bioinformatics metadata extensions."""
    if pd.isna(x) or str(x).strip() == "": 
        return False
    val = str(x).lower()
    return any(ext in val for ext in ['.csv', '.yaml', '.tsv', '.txt', '.gz'])

# --- 2. Process the Dataframe ---
# Assuming 'df' is your existing DataFrame
modalities_dict = {}

# We group by modality to ensure we capture unique files for each
for k, v in df.drop_duplicates('file_modality').groupby('file_modality'):
    row = v.iloc[0]
    modalities_dict[k] = {
        'R1_path': row['R1_path'],
        'R2_path': row['R2_path'],
        'barcode_onlist': row['barcode_onlist'] if sanitizing(row['barcode_onlist']) else None,
        'barcode_hashtag_map': row['barcode_hashtag_map'] if sanitizing(row['barcode_hashtag_map']) else None,
        'guide_design': row['guide_design'] if sanitizing(row['guide_design']) else None
    }

# --- 3. Generate the Shell Script ---
shell_script_name = "download_test_data.sh"
# Define 10MB in bytes
TEN_MB = 10485760

commands = [
    "#!/bin/bash",
    "# Auto-generated script for Perturb-seq test data download",
    "# Point to your credentials to avoid 401 Anonymous errors",
    'export GOOGLE_APPLICATION_CREDENTIALS="/Users/lf588/.config/gcloud/application_default_credentials.json"',
    "mkdir -p test_data",
    "echo 'Starting downloads into ./test_data...'\n"
]

for modality, info in modalities_dict.items():
    commands.append(f"### Modality: {modality} ###")
    
    # FastQ Chunks (10MB via range request)
    # Note: If these are .gz, the resulting file will be a truncated gzip
    r1_local = f"test_data/{modality}_R1_10mb.fastq.gz"
    r2_local = f"test_data/{modality}_R2_10mb.fastq.gz"
    
    commands.append(f"echo 'Fetching 10MB chunk for {modality} FastQs...'")
    commands.append(f"gsutil -o \"GSUtil:parallel_process_count=1\" cat -r 0-{TEN_MB} {info['R1_path']} > {r1_local}")
    commands.append(f"gsutil -o \"GSUtil:parallel_process_count=1\" cat -r 0-{TEN_MB} {info['R2_path']} > {r2_local}")

    # Metadata Files with specific naming tags
    metadata_mapping = {
        'barcode_onlist': 'barcodeWhitelist',
        'barcode_hashtag_map': 'hashMetadata',
        'guide_design': 'guideMetadata'
    }

    for key, tag in metadata_mapping.items():
        remote_path = info[key]
        if remote_path:
            ext = os.path.splitext(remote_path)[1]
            # Rename using the requested tags
            local_path = f"test_data/{modality}_{tag}{ext}"
            commands.append(f"echo 'Downloading {tag}: {os.path.basename(remote_path)}'")
            commands.append(f"gsutil -o \"GSUtil:parallel_process_count=1\" cp {remote_path} {local_path}")
    
    commands.append("") # Spacer

# --- 4. Write to File ---
with open(shell_script_name, 'w') as f:
    f.write("\n".join(commands))

os.chmod(shell_script_name, 0o755) 
print(f"Success! Run 'bash {shell_script_name}' to begin.")

Success! Run 'bash download_test_data.sh' to begin.


In [ ]:
!gsutil cat -r 0-5242880 gs://igvf-pertub-seq-pipeline-data/adam_test/Hon_WTC11-benchmark_TF-Perturb-seq/2025_08_14/IGVFFI2826BSFX.fastq > small_chunk.fastq

In [22]:
file = 'gs://igvf-pertub-seq-pipeline-data/scratch/bioinfolucas/new_03_seqspec/gary_IGVFDS4761PYUO_rna.yaml'
cmd = f'gcloud auth application-default login;gsutil -o "GSUtil:parallel_process_count=1" -m cp  -r {file}  . ' 
#write the command to a shell script
with open('download.sh', 'w') as f:
    f.write(cmd)




In [21]:
import os

# This tells gsutil exactly which credentials to use
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/lf588/.config/gcloud/application_default_credentials.json"

# Now run your command
!gsutil -o "GSUtil:parallel_process_count=1" cp -r gs://igvf-pertub-seq-pipeline-data/scratch/bioinfolucas/new_03_seqspec/gary_IGVFDS4761PYUO_rna.yaml .

ServiceException: 401 Anonymous caller does not have storage.objects.get access to the Google Cloud Storage object. Permission 'storage.objects.get' denied on resource (or it may not exist).


In [60]:
len('ACACTCTTTCCCTACACGACGCTCTTCCGATCT')

33

In [61]:
len('TCTTGTGGAAAGGACGAAACACCG')

24

In [77]:
'CACTCTCTATGTGCCTCTGTGTCTTTTTCACTGCTGCAGGGACTCAGAAAAAGAATTCAGGGACCCAAGAGACCCACTTTCTCAGTGGCTGATTCCCAGGCCCATCTCACCAGGCTGTGACTTCCTGTTTGCTGTGGGCAAGCCCCAGGGGACCCTCCCTACAGGTGGCAGGGCCTCCACTTCCCTTTGGCCAGTTGAGGCGGCTGGGAGGCGGAAGAATGAGGCATGAATGAAGGGGAATGAAGGGGGCGAGTAGCAGGACTCAATCTGCCTTTCATTCAACAAAATCTCTCTAATCCCCTCATTAGCTACCCAGCTGCCCAGACTGGCTTTCACTAGAGCATGT'[:251]

'CACTCTCTATGTGCCTCTGTGTCTTTTTCACTGCTGCAGGGACTCAGAAAAAGAATTCAGGGACCCAAGAGACCCACTTTCTCAGTGGCTGATTCCCAGGCCCATCTCACCAGGCTGTGACTTCCTGTTTGCTGTGGGCAAGCCCCAGGGGACCCTCCCTACAGGTGGCAGGGCCTCCACTTCCCTTTGGCCAGTTGAGGCGGCTGGGAGGCGGAAGAATGAGGCATGAATGAAGGGGAATGAAGGGGGCG'

In [71]:
'TTCTACTATTCTTTCCCCTGCACTGTACCCCCCAATCCCCCCTTTTCTTTTAAAATTGTGGATGAATACTGCCATTTGTCTCAAGATCTAGAATTCAAAAAAGCACCGACTCGGTGCCACTTTTTCAAGTTGATAACGGACTAGCCTTATTTTAACTTGCTATTTCTAGCTCTAAAAC'[:150]

'TTCTACTATTCTTTCCCCTGCACTGTACCCCCCAATCCCCCCTTTTCTTTTAAAATTGTGGATGAATACTGCCATTTGTCTCAAGATCTAGAATTCAAAAAAGCACCGACTCGGTGCCACTTTTTCAAGTTGATAACGGACTAGCCTTAT'

In [72]:
len('TTCTACTATTCTTTCCCCTGCACTGTACCCCCCAATCCCCCCTTTTCTTTTAAAATTGTGGATGAATACTGCCATTTGTCTCAAGATCTAGAATTCAAAAAAGCACCGACTCGGTGCCACTTTTTCAAGTTGATAACGGACTAGCCTTAT')

150